# Steel Beverage Can LCA

Cradle-to-grave carbon footprint of one 0.5 L steel beverage can, Brightway for the calculation, BAFU for the background. Fourth entry in the packaging comparison alongside the PET bottle, glass bottle, and aluminium can, all at the same 0.5 L functional unit.

## 1. Project setup

In [ ]:
import brightway2 as bw

bw.projects.set_current("Steel Can LCA")
bw.bw2setup()

print(list(bw.databases))

Creating default biosphere

Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.00 seconds
Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:49:55
  Finished: 08/06/2026 11:49:55
  Total time elapsed: 00:00:00
  CPU %: 97.50
  Memory %: 2.55
Created database: biosphere3
Creating default LCIA methods

Wrote 762 LCIA methods with 227223 characterization factors
Creating core data migrations

['biosphere3']


## 2. Import BAFU (ecoSpold1)

Same procedure as the other notebooks. Update the path below to the extracted ecoSpold1 folder.

In [2]:
import bw2io as bi

ECOSPOLD_PATH = "/Users/abhi/ecoSpold files"

importer = bi.SingleOutputEcospold1Importer(ECOSPOLD_PATH, "BAFU")

Extracting XML data from 11947 datasets


/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(

Extracted 11947 datasets in 6.69 seconds


In [3]:
fields_to_clean = ["name", "categories", "unit", "reference product", "location"]

for ds in importer.data:
    for field in fields_to_clean:
        if ds.get(field) is None:
            ds[field] = ""
    for exc in ds.get("exchanges", []):
        for field in fields_to_clean:
            if exc.get(field) is None:
                exc[field] = ""

In [4]:
importer.apply_strategies()
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 11 strategies in 8.64 seconds
Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
295213 unlinked exchanges
  Type biosphere: 2679 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 295213)

In [5]:
CATEGORY_FIX = {
    ("emissions to air",): ("air",),
    ("emissions to air", "high. pop."): ("air", "urban air close to ground"),
    ("emissions to air", "low. pop."): ("air", "non-urban air or from high stacks"),
    ("emissions to air", "stratosphere + troposphere"): ("air", "lower stratosphere + upper troposphere"),
    ("emissions to soil",): ("soil",),
    ("emissions to soil", "agricultural"): ("soil", "agricultural"),
    ("emissions to soil", "industrial"): ("soil", "industrial"),
    ("emissions to water",): ("water",),
    ("emissions to water", "ground-"): ("water", "ground-"),
    ("emissions to water", "ocean"): ("water", "ocean"),
    ("emissions to water", "river"): ("water", "surface water"),
    ("resources",): ("natural resource",),
    ("resources", "land"): ("natural resource", "land"),
    ("resources", "in ground"): ("natural resource", "in ground"),
    ("resources", "in air"): ("natural resource", "in air"),
    ("resources", "in water"): ("natural resource", "in water"),
    ("resources", "biotic"): ("natural resource", "biotic"),
}

fixed_count = 0
for ds in importer.data:
    for exc in ds.get("exchanges", []):
        if exc.get("type") == "biosphere":
            cats = tuple(exc.get("categories") or ())
            if cats in CATEGORY_FIX:
                exc["categories"] = CATEGORY_FIX[cats]
                fixed_count += 1

print(f"Rewrote categories on {fixed_count} exchanges")

Rewrote categories on 268970 exchanges


In [6]:
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
114060 unlinked exchanges
  Type biosphere: 1155 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 114060)

Check what's still unlinked before dropping it, same as before.

In [7]:
unlinked = list(importer.unlinked)
bio_unlinked = [u for u in unlinked if u.get("type") == "biosphere"]
tech_unlinked = [u for u in unlinked if u.get("type") == "technosphere"]

print(f"{len(bio_unlinked)} unique unlinked biosphere flows")
for u in bio_unlinked[:20]:
    print(u.get("name"), "|", u.get("categories"), "|", u.get("unit"))

print(f"\n{len(tech_unlinked)} unique unlinked technosphere")
for u in tech_unlinked[:24]:
    print(u.get("name"), "|", u.get("location"), "|", u.get("unit"))

1155 unique unlinked biosphere flows
NMVOC, non-methane volatile organic compounds, unspecified origin | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'non-urban air or from high stacks') | kilogram
Occupation, industrial area, built up | ('natural resource', 'land') | square meter-year
Transformation, to industrial area, built up | ('natural resource', 'land') | square meter
Particulates, < 2.5 um | ('air', 'urban air close to ground') | kilogram
Copper | ('air', 'urban air close to ground') | kilogram
Particulates, < 2.5 um | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'urban air close to ground') | kilogram
Selenium | ('air', 'non-urban air or from high stacks') | kilogram
Selenium | ('air', 'urban air close to ground') | kilogram
Nickel | ('air', 'non-urban air or from high stacks') | kilogram
Zinc | ('air', 'non-urban air or from high stacks') | kilogram
Nickel | ('air', 'urban air close to ground') | kilogram
Zinc | ('air'

In [8]:
importer.drop_unlinked(i_am_reckless=True)
importer.statistics()

Applying strategy: drop_unlinked
Applied 1 strategies in 0.09 seconds
11947 datasets
306003 exchanges
0 unlinked exchanges
  


(11947, 306003, 0)

In [9]:
importer.write_database()
print(list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:50:33
  Finished: 08/06/2026 11:50:41
  Total time elapsed: 00:00:08
  CPU %: 95.30
  Memory %: 2.07
Created database: BAFU
['biosphere3', 'BAFU']


## 3. Search BAFU for steel/tinplate production and can-forming

Run each search and look at the real results before picking anything.

In [10]:
ei = bw.Database("BAFU")

# General steel production
for r in ei.search("steel"):
    print(r, "|", r["location"])

'Steel, converter, chromium steel 18/8, at plant' (kilogram, RER, ['metals', 'ferro']) | RER
'Steel, electric, chromium steel 18/8, at plant' (kilogram, RER, ['metals', 'ferro']) | RER
'Powder coating, steel' (square meter, RER, ['metals', 'coating']) | RER
'Welding, arc, steel' (meter, RER, ['metals', 'welding']) | RER
'Sheet rolling, steel' (kilogram, RER, ['metals', 'chipless shaping']) | RER
'Recycling steel and iron' (kilogram, RER, ['recycling']) | RER
'Wire drawing, steel' (kilogram, RER, ['metals', 'chipless shaping']) | RER
'Welding, gas, steel' (meter, RER, ['metals', 'welding']) | RER
'Pipe, stainless steel' (kilogram, CH, ['heating', 'production of components']) | CH
'Hot rolling, steel' (kilogram, CH, ['metals', 'chipless shaping']) | CH
'Drawing of pipes, steel' (kilogram, RER, ['metals', 'chipless shaping']) | RER
'Hot rolling, steel' (kilogram, RER, ['metals', 'chipless shaping']) | RER
'Rolled steel, at regional storage' (kilogram, CH, ['metals', 'ferro']) | CH
'Steel,

In [11]:
# Tinplate specifically, food and beverage cans are almost always tin-coated
# steel rather than plain steel, worth checking if BAFU distinguishes it
for r in ei.search("tinplate"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("tin plated"):
    print(r, "|", r["location"])

---
'Tin plated chromium steel sheet, 2 mm, at plant' (square meter, RER, ['metals', 'non ferro']) | RER


In [12]:
# Can forming: deep drawing or stamping. Same expectation as the aluminium
# can notebook, BAFU may not have a can-specific process, a general
# sheet rolling or forming process is the closest proxy if so
for r in ei.search("deep drawing"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("sheet rolling steel"):
    print(r, "|", r["location"])

'Deep drawing, steel, 10000 kN press, automode operation' (kilogram, RER, ['metals', 'chipless shaping']) | RER
'xx Deep drawing, steel, 38000 kN press, automode operation' (kilogram, RER, ['processing, obsolete', 'metals\\chipless shaping']) | RER
'xx Deep drawing, steel, 3500 kN press, automode operation' (kilogram, RER, ['processing, obsolete', 'metals\\chipless shaping']) | RER
'xx Deep drawing, steel, 650 kN press, automode operation' (kilogram, RER, ['processing, obsolete', 'metals\\chipless shaping']) | RER
'xx Deep drawing, steel, 38000 kN press, single stroke operation' (kilogram, RER, ['processing, obsolete', 'metals\\chipless shaping']) | RER
'xx Deep drawing, steel, 3500 kN press, single stroke operation' (kilogram, RER, ['processing, obsolete', 'metals\\chipless shaping']) | RER
'xx Deep drawing, steel, 650 kN press, single stroke operation' (kilogram, RER, ['processing, obsolete', 'metals\\chipless shaping']) | RER
'xx Deep drawing, steel, 10000 kN press, single stroke op

Fill in the picks below once the searches above have been run and the correct index is known for each.

In [13]:
# Replace the "..." search terms and [0] indices with the confirmed picks
# steel_material = ei.search("...")[0]
# can_forming = ei.search("...")[0]

The transport and disposal picks below were seen at these positions in earlier notebooks, transport in the PET bottle and aluminium can models, steel disposal in the PET bottle's `municipal solid waste incineration` search. Print and check them before trusting the index, same caution as always.

In [14]:
truck = ei.search("transport lorry")[2]
print(truck)  # expect: 'Transport, freight, lorry, fleet average' (ton kilometer, CH)

incineration = ei.search("municipal solid waste incineration")[10]
print(incineration)  # expect: 'Disposal, steel, 0% water, to municipal incineration' (CH)

'Transport, freight, lorry, fleet average' (ton kilometer, CH, ['transport systems', 'road\\Transformation'])
'Disposal, steel, 0% water, to municipal incineration' (kilogram, CH, ['waste management', 'municipal incineration'])


## 4. Recycled content and end-of-life assumptions

Steel's recycling rate is typically even higher than aluminium's where collection systems exist, since magnetic separation makes it cheap to pull out of mixed waste. I'm modelling end-of-life as mostly recycling with a smaller incineration share, and skipping landfill entirely as negligible, unlike the aluminium can and PET bottle notebooks where all three pathways were modelled. If a landfill share turns out to matter for a specific region, add it back the same way those notebooks did.

There's no confirmed BAFU steel recycling activity yet (see section 3), so for now the recycled share is routed through the same incineration pick as a placeholder, same workaround used in the aluminium can notebook until `secondary` material or recycling picks are confirmed.

In [15]:
# ---- Physical parameters ----
CAN_MASS_KG = 0.045   # a 0.5 L steel beverage/food can, body and lid, weighs more than
                       # an aluminium can due to steel's higher density and thicker walls
DIST_KM = 100          # factory to retailer distance, same assumption as the other notebooks

# ---- End-of-life split ----
EoL_RECYCLE = 0.80     # steel's magnetic separability pushes recycling rates high
EoL_INCIN = 0.20

print("Assumptions set.")

Assumptions set.


## 5. Build the foreground activity

Same simplified-until-confirmed pattern as the aluminium can notebook. Once `steel_material` and `can_forming` are confirmed, uncomment those lines and rerun.

In [ ]:
DB_NAME = "Steel Can"

if DB_NAME in bw.databases:
    del bw.databases[DB_NAME]

exchanges = [
    {"input": (DB_NAME, "steel_can_0.5L"), "amount": 1, "unit": "piece", "type": "production"},

    # ---- Material stage ----
    #{"input": steel_material.key, "amount": CAN_MASS_KG, "unit": "kilogram", "type": "technosphere"},

    # ---- Can forming ----
    # {"input": can_forming.key, "amount": CAN_MASS_KG, "unit": "kilogram", "type": "technosphere"},

    # ---- Distribution ----
    {"input": truck.key, "amount": CAN_MASS_KG * DIST_KM / 1000, "unit": "ton kilometer", "type": "technosphere"},

    # ---- End of life ----
    # Recycled share routed through incineration as a placeholder, see note above
    {"input": incineration.key, "amount": CAN_MASS_KG * (EoL_RECYCLE + EoL_INCIN), "unit": "kilogram", "type": "technosphere"},
]

foreground = bw.Database(DB_NAME)
foreground.write({
    (DB_NAME, "steel_can_0.5L"): {
        "name": "Steel can, 0.5 litre, cradle-to-grave",
        "reference product": "Steel can, 0.5 litre, cradle-to-grave",
        "unit": "piece",
        "location": "CH",
        "type": "process",
        "exchanges": exchanges,
    }
})

print("Database written:", list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:51:04
  Finished: 08/06/2026 11:51:04
  Total time elapsed: 00:00:00
  CPU %: 40.20
  Memory %: 2.26
Database written: ['biosphere3', 'BAFU', 'Steel Can']


Note to self: until the material and forming picks are uncommented, this model only accounts for transport and end-of-life. The score in section 6 will be far too low until the steel material stage is filled in.

## 6. Run the LCA

Same method as the other notebooks.

In [17]:
can = bw.get_activity((DB_NAME, "steel_can_0.5L"))

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

lca = bw.LCA({can: 1}, method)
lca.lci()
lca.lcia()

print(f"GWP 100a: {lca.score:.4f} kg CO2-eq per can")

GWP 100a: 0.0013 kg CO2-eq per can


## 7. Monte Carlo uncertainty analysis

The single `GWP 100a` score above is one point estimate, it hides how much the result could vary given real uncertainty in every exchange behind it. Monte Carlo reruns the calculation many times, resampling every exchange from its uncertainty distribution each time, and returns a spread instead of one number.

Two things to know first: `bw2calc`'s Monte Carlo solver can raise `ValueError: 'scipy.sparse.linalg.cgs' called with invalid atol='legacy'` on newer `scipy` versions, if that happens, run `conda install "scipy<1.12"` in the `ab` environment and restart the kernel. And Monte Carlo only produces a real spread if the exchanges being sampled actually carry uncertainty data, the foreground exchanges written here don't have any set by default, so they'll stay fixed across iterations unless it's added explicitly.

In [18]:
from bw2calc import MonteCarloLCA

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

mc = MonteCarloLCA({can: 1}, method)
n_iterations = 1000
scores = [next(mc) for _ in range(n_iterations)]

import numpy as np

print(f"Iterations: {n_iterations}")
print(f"Mean:   {np.mean(scores):.4f} kg CO2-eq per can")
print(f"Median: {np.median(scores):.4f} kg CO2-eq per can")
print(f"Std dev: {np.std(scores):.4f}")
print(f"5th to 95th percentile: {np.percentile(scores, 5):.4f} to {np.percentile(scores, 95):.4f} kg CO2-eq")

Iterations: 1000
Mean:   0.0014 kg CO2-eq per can
Median: 0.0014 kg CO2-eq per can
Std dev: 0.0001
5th to 95th percentile: 0.0012 to 0.0015 kg CO2-eq
